# LeetCode 1424 — Diagonal Traverse II bằng Python

Link đề: https://leetcode.com/problems/diagonal-traverse-ii/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Heap (Priority Queue) (thêm: Hash Table, BFS)

Skeleton LeetCode:

```python
class Solution:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra **đặc trưng của đường chéo phụ**: mọi ô `(i, j)` cùng đường chéo có chung `i + j`.
2. Làm việc với **mảng răng cưa** (jagged array — các hàng dài khác nhau) mà không giả định ma trận chữ nhật.
3. Đi từ cách quét "ma trận ảo" chậm `O(R · D)` đến các cách **chỉ chạm vào ô thật**: gom nhóm theo `i + j` (`O(N)`), sắp xếp theo khoá `(i + j, −i)` (`O(N log N)`), BFS từ `(0, 0)` (`O(N)`).
4. Biết chọn **khoá sắp xếp** đúng (dấu trừ để đảo chiều) và cách duyệt hàng **từ dưới lên** để có thứ tự trong mỗi đường chéo.
5. Tránh lỗi: dùng `len(nums[0])` làm số cột, sai chiều trong đường chéo, BFS thêm trùng ô.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `nums` (các hàng **có thể dài khác nhau**). Trả về mọi phần tử theo **thứ tự đường chéo**: lần lượt từng đường chéo phụ (từ góc trên-trái đi ra), trong mỗi đường chéo đi **từ dưới-trái lên trên-phải**.

**Ví dụ 1:** `nums = [[1,2,3],[4,5,6],[7,8,9]]` → `[1,4,2,7,5,3,8,6,9]`.

**Ví dụ 2:** `nums = [[1,2,3,4,5],[6,7],[8],[9,10,11],[12,13,14,15,16]]` → `[1,6,2,8,7,3,9,4,12,10,5,13,11,14,15,16]`.

**Ràng buộc:** `1 <= nums.length <= 10^5`; `1 <= nums[i].length <= 10^5`; `1 <= tổng mọi nums[i].length <= 10^5`; `1 <= nums[i][j] <= 10^5`.

Hình dung ví dụ 2 (ô trống là không tồn tại):

```
 1  2  3  4  5
 6  7
 8
 9 10 11
12 13 14 15 16
```

In [1]:
import random
import time
import heapq
from collections import defaultdict, deque
from typing import List

ex1 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
ex2 = [[1, 2, 3, 4, 5], [6, 7], [8], [9, 10, 11], [12, 13, 14, 15, 16]]
examples = [(ex1, [1, 4, 2, 7, 5, 3, 8, 6, 9]),
            (ex2, [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16])]
for nums, want in examples:
    print(nums, "->", want)

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] -> [1, 4, 2, 7, 5, 3, 8, 6, 9]
[[1, 2, 3, 4, 5], [6, 7], [8], [9, 10, 11], [12, 13, 14, 15, 16]] -> [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


## 3. Kiến thức nền

### 3.1 Mảng răng cưa trong Python

`list` của `list`, mỗi hàng một độ dài. Truy cập `nums[i][j]` chỉ hợp lệ khi `j < len(nums[i])`. Duyệt an toàn: `for i, row in enumerate(nums): for j, x in enumerate(row)`.

In [2]:
for i, row in enumerate(ex2):
    print(i, [(i, j, x) for j, x in enumerate(row)])

0 [(0, 0, 1), (0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 5)]
1 [(1, 0, 6), (1, 1, 7)]
2 [(2, 0, 8)]
3 [(3, 0, 9), (3, 1, 10), (3, 2, 11)]
4 [(4, 0, 12), (4, 1, 13), (4, 2, 14), (4, 3, 15), (4, 4, 16)]


### 3.2 `defaultdict(list)` để gom nhóm

`groups[key].append(x)` tự tạo list rỗng cho khoá mới. Dùng để gom các ô theo `i + j`.

In [3]:
groups = defaultdict(list)
for i, row in enumerate(ex1):
    for j, x in enumerate(row):
        groups[i + j].append(x)
print(dict(groups))

{0: [1], 1: [2, 4], 2: [3, 5, 7], 3: [6, 8], 4: [9]}


### 3.3 Sắp xếp theo tuple và dấu trừ để đảo chiều

Python so sánh tuple theo từng thành phần. Khoá `(d, -i)` nghĩa là: tăng theo `d`, nếu hoà thì **giảm** theo `i` (vì `-i` tăng khi `i` giảm).

In [4]:
cells = [(0, 1), (1, 0), (0, 0), (2, 0), (1, 1), (0, 2)]
print(sorted(cells, key=lambda c: (c[0] + c[1], -c[0])))

[(0, 0), (1, 0), (0, 1), (2, 0), (1, 1), (0, 2)]


### 3.4 `deque` cho BFS

Hàng đợi: `append` ở cuối, `popleft` ở đầu, đều `O(1)`. BFS xử lý các nút theo **tầng** — ở đây tầng chính là đường chéo `i + j`.

In [5]:
q = deque([(0, 0)])
q.append((1, 0)); q.append((0, 1))
print(q.popleft(), list(q))

(0, 0) [(1, 0), (0, 1)]


### 3.5 Duyệt ngược `reversed(range(...))`

Để duyệt hàng **từ dưới lên**: `for i in range(len(nums) - 1, -1, -1)` hoặc `for i in reversed(range(len(nums)))`.

In [6]:
print(list(range(4, -1, -1)), list(reversed(range(5))))

[4, 3, 2, 1, 0] [4, 3, 2, 1, 0]


## 4. Ý tưởng nền tảng / trực giác

Ghi `i + j` vào từng ô của ví dụ 1:

```
(0)1  (1)2  (2)3
(1)4  (2)5  (3)6
(2)7  (3)8  (4)9
```

Các ô cùng số trong ngoặc nằm trên cùng một đường chéo phụ. Đề yêu cầu duyệt đường chéo `0, 1, 2, 3, 4`, và trong mỗi đường chéo đi từ **hàng lớn** đến **hàng nhỏ** (dưới lên trên):
- `d=0`: `1`
- `d=1`: `(1,0)=4`, `(0,1)=2`
- `d=2`: `7, 5, 3`
- `d=3`: `8, 6`
- `d=4`: `9`

⇒ `[1, 4, 2, 7, 5, 3, 8, 6, 9]` ✔.

Với mảng răng cưa, một số ô "vắng mặt" — ta chỉ bỏ qua chúng. Câu hỏi là làm sao **không phải đi qua các ô vắng** (có thể rất nhiều).

In [7]:
for i, row in enumerate(ex2):
    print("  ".join(f"{x:2d}(d={i + j})" for j, x in enumerate(row)))

 1(d=0)   2(d=1)   3(d=2)   4(d=3)   5(d=4)
 6(d=1)   7(d=2)
 8(d=2)
 9(d=3)  10(d=4)  11(d=5)
12(d=4)  13(d=5)  14(d=6)  15(d=7)  16(d=8)


## 5. Cách 1 — Brute Force / baseline: quét như ma trận chữ nhật ảo

**Ý tưởng:** gọi `D` = chỉ số đường chéo lớn nhất `= max(i + len(nums[i]) − 1)`. Với mỗi `d` từ `0` đến `D`, duyệt `i` từ `min(d, R−1)` **xuống** `0`, `j = d − i`, nếu `j < len(nums[i])` thì lấy.

In [8]:
def diag_brute(nums):
    R = len(nums)
    D = max(i + len(row) - 1 for i, row in enumerate(nums))
    res = []
    for d in range(D + 1):
        for i in range(min(d, R - 1), -1, -1):
            j = d - i
            if j < len(nums[i]):
                res.append(nums[i][j])
    return res

for nums, want in examples:
    print(diag_brute(nums) == want)

True
True


**Độ phức tạp:** `O(R · D)` lần kiểm tra — phần lớn là ô **không tồn tại**. Trường hợp xấu: `10^5` hàng, mỗi hàng 1 phần tử ⇒ `D ≈ 10^5`, vòng lặp `≈ R·D/2 = 5·10^9` ⇒ **TLE**. Hoặc 2 hàng: hàng đầu dài `5·10^4` và một hàng dài `5·10^4` ở hàng thứ `5·10^4`… cũng tệ.

**Nút thắt:** đi qua các ô ảo. Ta cần cách chỉ chạm vào `N = tổng số phần tử` ô thật.

In [9]:
tall = [[i] for i in range(3000)]           # 3000 hàng, mỗi hàng 1 phần tử
t = time.perf_counter(); diag_brute(tall); el = time.perf_counter() - t
print(f"brute với R=3000 hàng, N=3000 phần tử: {el:.3f}s (R=10^5 sẽ chậm hơn ~1000 lần)")

brute với R=3000 hàng, N=3000 phần tử: 0.176s (R=10^5 sẽ chậm hơn ~1000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — khoá đường chéo `d = i + j`:** chỉ cần duyệt mỗi ô thật **một lần** và đặt nó vào nhóm `d`. Thứ tự đầu ra là: nhóm `d` tăng dần; trong nhóm, `i` **giảm** dần.

**Insight 2 — thứ tự trong nhóm miễn phí:** **nếu** ta duyệt các hàng **từ dưới lên** (i giảm) **thì** mỗi nhóm nhận phần tử đúng theo thứ tự `i` giảm ⇒ không cần sort trong nhóm. (Hoặc duyệt từ trên xuống rồi đảo từng nhóm.)

**Insight 3 — số nhóm ≤ N:** chỉ số nhóm lớn nhất `D ≤ R − 1 + max_len − 1 < N + N`, nhưng chỉ những nhóm có phần tử mới tồn tại; dùng `dict` hoặc list dài `D + 1`.

**Insight 4 — BFS:** coi ô `(i, j)` có "con" là `(i, j+1)` (bên phải) và — chỉ khi `j == 0` — `(i+1, 0)` (bên dưới). Mỗi ô có đúng một cha ⇒ là một **cây**, độ sâu của ô = `i + j`. BFS theo tầng cho đúng thứ tự đường chéo; đẩy con "dưới" trước con "phải" để trong tầng đi từ dưới lên.

## 7. Các cách tiếp theo

### Cách 2 — Gom nhóm theo `i + j`, duyệt hàng từ dưới lên

**Từng đoạn:**
1. `groups = defaultdict(list)`.
2. `for i in reversed(range(R))`: `for j, x in enumerate(nums[i])`: `groups[i + j].append(x)`.
3. Nối các nhóm theo `d = 0, 1, 2, …` (khoá liên tục từ `0` tới `max`, vì ô `(0,0)` luôn có và đường chéo không bị "đứt").

In [10]:
def diag_group(nums):
    groups = defaultdict(list)
    for i in range(len(nums) - 1, -1, -1):        # dưới lên
        for j, x in enumerate(nums[i]):
            groups[i + j].append(x)
    res = []
    for d in range(len(groups)):
        res.extend(groups[d])
    return res

for nums, want in examples:
    print(diag_group(nums) == want)

True
True


**Vì sao `range(len(groups))` đủ?** Nếu ô `(i, j)` tồn tại thì mọi ô `(i, 0..j)` và `(0, 0)` đều tồn tại… nhưng ô `(i−1, …)` có thể không. Tuy vậy, mọi `d` từ `0` tới `i + j` đều có ít nhất ô `(i, d − i)` nếu `d ≥ i`, hoặc ô `(d, 0)` nếu `d < i` (hàng `d < i` luôn tồn tại và có phần tử thứ 0). Vậy các khoá liên tục `0..D`.

In [11]:
class SolutionGroup:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        groups = defaultdict(list)
        for i in range(len(nums) - 1, -1, -1):
            for j, x in enumerate(nums[i]):
                groups[i + j].append(x)
        res = []
        for d in range(len(groups)):
            res.extend(groups[d])
        return res

print(SolutionGroup().findDiagonalOrder(ex2))

[1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


**Độ phức tạp:** `O(N + R)` thời gian (mỗi ô một lần, cộng duyệt các hàng), `O(N)` bộ nhớ.

### Cách 3 — Sắp xếp theo khoá `(i + j, −i)`

**Từng đoạn:**
1. Tạo list bộ ba `(i + j, -i, x)` cho mọi ô.
2. `sort()` — Python so tuple: `d` tăng, hoà thì `-i` tăng (tức `i` giảm).
3. Lấy thành phần `x`.

Đây là gợi ý chính thức của LeetCode — dễ nghĩ, ít bẫy.

In [12]:
def diag_sort(nums):
    cells = [(i + j, -i, x) for i, row in enumerate(nums) for j, x in enumerate(row)]
    cells.sort()
    return [x for _, _, x in cells]

for nums, want in examples:
    print(diag_sort(nums) == want)

True
True


In [13]:
class SolutionSort:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        cells = [(i + j, -i, x) for i, row in enumerate(nums) for j, x in enumerate(row)]
        cells.sort()
        return [x for _, _, x in cells]

print(SolutionSort().findDiagonalOrder(ex1))

[1, 4, 2, 7, 5, 3, 8, 6, 9]


**Độ phức tạp:** `O(N log N)` thời gian, `O(N)` bộ nhớ. (Biến thể dùng **heap** — chủ đề "Heap" trong tag — cũng `O(N log N)`: đẩy mọi bộ ba vào heap rồi pop dần.)

In [14]:
def diag_heap(nums):
    h = [(i + j, -i, x) for i, row in enumerate(nums) for j, x in enumerate(row)]
    heapq.heapify(h)
    return [heapq.heappop(h)[2] for _ in range(len(h))]

print([diag_heap(nums) == want for nums, want in examples])

[True, True]


### Cách 4 — BFS từ `(0, 0)`

**Từng đoạn:**
1. `q = deque([(0, 0)])`.
2. Lấy `(i, j)` ra, ghi `nums[i][j]`.
3. **Nếu** `j == 0` và hàng `i + 1` tồn tại ⇒ đẩy `(i + 1, 0)` (con dưới) — **trước**.
4. **Nếu** `j + 1 < len(nums[i])` ⇒ đẩy `(i, j + 1)` (con phải).

Chỉ cột 0 mới sinh con dưới ⇒ mỗi ô được đẩy **đúng một lần** (không cần `visited`).

In [15]:
def diag_bfs(nums):
    R = len(nums)
    res = []
    q = deque([(0, 0)])
    while q:
        i, j = q.popleft()
        res.append(nums[i][j])
        if j == 0 and i + 1 < R:
            q.append((i + 1, 0))
        if j + 1 < len(nums[i]):
            q.append((i, j + 1))
    return res

for nums, want in examples:
    print(diag_bfs(nums) == want)

True
True


In [16]:
class SolutionBFS:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        R = len(nums)
        res = []
        q = deque([(0, 0)])
        while q:
            i, j = q.popleft()
            res.append(nums[i][j])
            if j == 0 and i + 1 < R:
                q.append((i + 1, 0))
            if j + 1 < len(nums[i]):
                q.append((i, j + 1))
        return res

print(SolutionBFS().findDiagonalOrder(ex2))

[1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


**Độ phức tạp:** `O(N)` thời gian, `O(N)` bộ nhớ trong trường hợp xấu (hàng đợi chứa cả một đường chéo, thường nhỏ hơn nhiều).

**Vì sao thứ tự trong tầng là dưới-lên?** Bằng quy nạp: tầng `d` trong hàng đợi đang xếp theo `i` giảm. Các con của chúng: ô `(i, j)` sinh `(i+1, 0)` (chỉ khi `j=0`, đó là ô có `i` lớn nhất của tầng, nằm **đầu** tầng) rồi `(i, j+1)`. Vì vậy tầng `d+1` nhận `(d+1, 0)` trước, sau đó các `(i, j+1)` theo `i` giảm ⇒ vẫn `i` giảm.

## 8. TRACE từng bước

Trace cách 4 (BFS) trên ví dụ 2 — in hàng đợi sau mỗi lần lấy ra:

| bước | lấy ra `(i,j)` | giá trị | đẩy vào | hàng đợi sau |
|---|---|---|---|---|
| 1 | (0,0) | 1 | (1,0), (0,1) | (1,0) (0,1) |
| 2 | (1,0) | 6 | (2,0), (1,1) | (0,1) (2,0) (1,1) |
| 3 | (0,1) | 2 | (0,2) | (2,0) (1,1) (0,2) |
| 4 | (2,0) | 8 | (3,0) | (1,1) (0,2) (3,0) |
| … | … | … | … | … |

In [17]:
def trace_bfs(nums, limit=None):
    R = len(nums)
    res = []
    q = deque([(0, 0)])
    step = 0
    while q:
        i, j = q.popleft()
        res.append(nums[i][j])
        pushed = []
        if j == 0 and i + 1 < R:
            q.append((i + 1, 0)); pushed.append((i + 1, 0))
        if j + 1 < len(nums[i]):
            q.append((i, j + 1)); pushed.append((i, j + 1))
        step += 1
        if limit is None or step <= limit:
            print(f"bước {step:2d}: lấy ({i},{j}) d={i + j} giá trị {nums[i][j]:2d} | đẩy {pushed} | q={list(q)}")
    print("kết quả:", res)
    return res

_ = trace_bfs(ex2)

bước  1: lấy (0,0) d=0 giá trị  1 | đẩy [(1, 0), (0, 1)] | q=[(1, 0), (0, 1)]
bước  2: lấy (1,0) d=1 giá trị  6 | đẩy [(2, 0), (1, 1)] | q=[(0, 1), (2, 0), (1, 1)]
bước  3: lấy (0,1) d=1 giá trị  2 | đẩy [(0, 2)] | q=[(2, 0), (1, 1), (0, 2)]
bước  4: lấy (2,0) d=2 giá trị  8 | đẩy [(3, 0)] | q=[(1, 1), (0, 2), (3, 0)]
bước  5: lấy (1,1) d=2 giá trị  7 | đẩy [] | q=[(0, 2), (3, 0)]
bước  6: lấy (0,2) d=2 giá trị  3 | đẩy [(0, 3)] | q=[(3, 0), (0, 3)]
bước  7: lấy (3,0) d=3 giá trị  9 | đẩy [(4, 0), (3, 1)] | q=[(0, 3), (4, 0), (3, 1)]
bước  8: lấy (0,3) d=3 giá trị  4 | đẩy [(0, 4)] | q=[(4, 0), (3, 1), (0, 4)]
bước  9: lấy (4,0) d=4 giá trị 12 | đẩy [(4, 1)] | q=[(3, 1), (0, 4), (4, 1)]
bước 10: lấy (3,1) d=4 giá trị 10 | đẩy [(3, 2)] | q=[(0, 4), (4, 1), (3, 2)]
bước 11: lấy (0,4) d=4 giá trị  5 | đẩy [] | q=[(4, 1), (3, 2)]
bước 12: lấy (4,1) d=5 giá trị 13 | đẩy [(4, 2)] | q=[(3, 2), (4, 2)]
bước 13: lấy (3,2) d=5 giá trị 11 | đẩy [] | q=[(4, 2)]
bước 14: lấy (4,2) d=6 giá trị 14 | 

Trace cách 2 (gom nhóm) trên ví dụ 2 — duyệt hàng dưới lên:

In [18]:
def trace_group(nums):
    groups = defaultdict(list)
    for i in range(len(nums) - 1, -1, -1):
        for j, x in enumerate(nums[i]):
            groups[i + j].append(x)
        print(f"sau hàng {i}: " + ", ".join(f"d{d}={groups[d]}" for d in sorted(groups)))
    res = [x for d in range(len(groups)) for x in groups[d]]
    print("nối các nhóm:", res)
    return res

_ = trace_group(ex2)

sau hàng 4: d4=[12], d5=[13], d6=[14], d7=[15], d8=[16]
sau hàng 3: d3=[9], d4=[12, 10], d5=[13, 11], d6=[14], d7=[15], d8=[16]
sau hàng 2: d2=[8], d3=[9], d4=[12, 10], d5=[13, 11], d6=[14], d7=[15], d8=[16]
sau hàng 1: d1=[6], d2=[8, 7], d3=[9], d4=[12, 10], d5=[13, 11], d6=[14], d7=[15], d8=[16]
sau hàng 0: d0=[1], d1=[6, 2], d2=[8, 7, 3], d3=[9, 4], d4=[12, 10, 5], d5=[13, 11], d6=[14], d7=[15], d8=[16]
nối các nhóm: [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


## 9. Code nộp LeetCode

Dùng cách 2 (gom nhóm theo `i + j`, duyệt hàng từ dưới lên) — `O(N)`, ngắn gọn:

In [19]:
class Solution:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        groups = defaultdict(list)                 # groups[d] = các phần tử trên đường chéo d = i + j
        for i in range(len(nums) - 1, -1, -1):     # hàng dưới trước -> trong nhóm đã đúng thứ tự
            for j, x in enumerate(nums[i]):
                groups[i + j].append(x)
        res = []
        for d in range(len(groups)):               # các d liên tục 0..max
            res.extend(groups[d])
        return res

for nums, want in examples:
    assert Solution().findDiagonalOrder(nums) == want
print("2 ví dụ: OK")

2 ví dụ: OK


## 10. Vì sao thuật toán đúng

- **Thứ tự cần đạt** là thứ tự tăng của khoá `(i + j, −i)`: đường chéo trước, trong đường chéo hàng lớn trước. Các ô khác nhau có khoá khác nhau (biết `i` và `i + j` thì biết `j`), nên thứ tự là **duy nhất**.
- **Cách 3** sắp đúng theo khoá đó ⇒ đúng ngay.
- **Cách 2:** mỗi ô vào đúng nhóm `d = i + j`. Vì duyệt `i` giảm dần, phần tử được `append` vào nhóm theo `i` giảm ⇒ mỗi nhóm đúng thứ tự `−i` tăng. Nối nhóm theo `d` tăng ⇒ đúng thứ tự khoá. Các `d` liên tục từ `0` tới `D` (lập luận ở cách 2) nên không bỏ sót nhóm.
- **Cách 4:** đồ thị "con phải / con dưới (chỉ cột 0)" là một cây gốc `(0,0)` phủ mọi ô (ô `(i, j)` đến được qua `(0,0)→(1,0)→…→(i,0)→(i,1)→…→(i,j)`), độ sâu = `i + j`. BFS lấy ra theo độ sâu tăng; bất biến "trong mỗi tầng, `i` giảm" được giữ như đã chứng minh ở mục 7.

## 11. Phân tích Time / Space

Gọi `N` = tổng số phần tử (`≤ 10^5`), `R` = số hàng, `D` = chỉ số đường chéo lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Quét ma trận ảo | `O(R · D)` (có thể `~10^10`) | `O(N)` (kết quả) |
| 2. Gom nhóm `i + j` | `O(N + R)` | `O(N)` |
| 3. Sort `(i+j, −i)` / heap | `O(N log N)` | `O(N)` |
| 4. BFS | `O(N)` | `O(N)` |

Lưu ý `R ≤ N` vì mỗi hàng có ít nhất 1 phần tử.

In [20]:
rng = random.Random(1424)
def rand_jagged(rng, N, max_rows):
    rows = rng.randint(1, min(N, max_rows))
    lens = [1] * rows
    for _ in range(N - rows):
        lens[rng.randrange(rows)] += 1
    return [[rng.randint(1, 10**5) for _ in range(L)] for L in lens]

cases = {"10^5 hàng x 1": [[i + 1] for i in range(10**5)],
         "1 hàng x 10^5": [list(range(1, 10**5 + 1))],
         "ngẫu nhiên": rand_jagged(rng, 10**5, 2000)}
for cname, nums in cases.items():
    row = []
    for name, f in [("group", diag_group), ("sort", diag_sort), ("bfs", diag_bfs)]:
        t = time.perf_counter(); f(nums); row.append(f"{name}={time.perf_counter() - t:.3f}s")
    print(f"{cname:14s}: " + ", ".join(row))

10^5 hàng x 1 : group=0.074s, sort=0.027s, bfs=0.013s


1 hàng x 10^5 : group=0.028s, sort=0.043s, bfs=0.011s
ngẫu nhiên    : group=0.007s, sort=0.027s, bfs=0.013s


## 12. Bộ test

1. Hai ví dụ của đề.
2. Edge: một ô; một hàng; một cột; hàng giữa rất ngắn còn hàng dưới rất dài; hàng đầu ngắn.
3. Đối chiếu mọi cách với brute force trên mảng răng cưa ngẫu nhiên (seed cố định).
4. Tính chất: kết quả là một hoán vị của mọi phần tử (đối chiếu sau khi sort).

In [21]:
impls = {
    "brute": diag_brute, "group": diag_group, "sort": diag_sort, "heap": diag_heap, "bfs": diag_bfs,
    "Solution": lambda a: Solution().findDiagonalOrder(a),
    "SolSort": lambda a: SolutionSort().findDiagonalOrder(a),
    "SolBFS": lambda a: SolutionBFS().findDiagonalOrder(a),
}
edge = [([[7]], [7]),
        ([[1, 2, 3, 4]], [1, 2, 3, 4]),
        ([[1], [2], [3]], [1, 2, 3]),
        ([[1, 2], [3], [4, 5, 6, 7]], [1, 3, 2, 4, 5, 6, 7]),
        ([[1], [2, 3, 4], [5, 6]], [1, 2, 5, 3, 6, 4]),
        ([[1, 2, 3], [4], [5], [6, 7, 8, 9]], [1, 4, 2, 5, 3, 6, 7, 8, 9])]
for name, f in impls.items():
    for nums, want in examples + edge:
        assert f(nums) == want, (name, nums, f(nums))
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [22]:
rng = random.Random(42)
for _ in range(400):
    N = rng.randint(1, 40)
    nums = rand_jagged(rng, N, 12)
    ref = diag_brute(nums)
    assert sorted(ref) == sorted(x for row in nums for x in row)
    for name, f in impls.items():
        assert f(nums) == ref, (name, nums)
print("random 400 mảng răng cưa: OK")

random 400 mảng răng cưa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi như ma trận chữ nhật (`cols = len(nums[0])`)

In [23]:
def wrong_rectangular(nums):
    R, Cc = len(nums), len(nums[0])                 # SAI: các hàng dài khác nhau
    res = []
    for d in range(R + Cc - 1):
        for i in range(min(d, R - 1), -1, -1):
            j = d - i
            if j < Cc:
                if j < len(nums[i]):               # thêm kiểm tra để không văng IndexError
                    res.append(nums[i][j])
                else:
                    res.append("<?>")              # đánh dấu ô không tồn tại mà code sai cố đọc
    return res

got = wrong_rectangular(ex2)
print("sai :", got)
print("đúng:", Solution().findDiagonalOrder(ex2))

sai : [1, 6, 2, 8, 7, 3, 9, '<?>', '<?>', 4, 12, 10, '<?>', '<?>', 5, 13, 11, '<?>', '<?>', 14, '<?>', '<?>', 15, '<?>', 16]
đúng: [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


**Vì sao sai:** hàng 0 của ví dụ 2 dài 5 nhưng hàng 2 chỉ dài 1 và hàng 4 dài 5. Lấy `len(nums[0])` làm số cột vừa cố đọc ô không tồn tại (code thật sẽ `IndexError`, ở đây in `<?>`), vừa có thể bỏ sót phần tử ở hàng dài hơn hàng 0.

**Sửa:** chỉ duyệt các ô thật `for j, x in enumerate(nums[i])`.

### Lỗi 2 — Duyệt hàng từ trên xuống mà không đảo nhóm

In [24]:
def wrong_top_down(nums):
    groups = defaultdict(list)
    for i, row in enumerate(nums):                 # SAI: trên xuống -> trong nhóm i tăng
        for j, x in enumerate(row):
            groups[i + j].append(x)
    return [x for d in range(len(groups)) for x in groups[d]]

print("sai :", wrong_top_down(ex1))
print("đúng:", Solution().findDiagonalOrder(ex1))

sai : [1, 2, 4, 3, 5, 7, 6, 8, 9]
đúng: [1, 4, 2, 7, 5, 3, 8, 6, 9]


**Vì sao sai:** trong mỗi đường chéo phải đi từ **dưới lên** (hàng lớn trước). Duyệt từ trên xuống cho thứ tự ngược lại: `[1, 2, 4, 3, 5, 7, …]`.

**Sửa:** duyệt `i` giảm dần, hoặc `groups[d].reverse()` trước khi nối.

### Lỗi 3 — Sai khoá sắp xếp `(i + j, i)` thay vì `(i + j, −i)`

In [25]:
def wrong_key(nums):
    cells = [(i + j, i, x) for i, row in enumerate(nums) for j, x in enumerate(row)]   # SAI
    return [x for _, _, x in sorted(cells)]

print("sai :", wrong_key(ex2))
print("đúng:", Solution().findDiagonalOrder(ex2))

sai : [1, 2, 6, 3, 7, 8, 4, 9, 5, 10, 12, 11, 13, 14, 15, 16]
đúng: [1, 6, 2, 8, 7, 3, 9, 4, 12, 10, 5, 13, 11, 14, 15, 16]


**Vì sao sai:** hoà `d` thì phải ưu tiên `i` **lớn**. Khoá `i` tăng làm đảo chiều đường chéo. Một lỗi tương tự: sort theo `(i + j, x)` — so theo **giá trị** chứ không theo vị trí, sai ngay khi giá trị không tăng dần theo hàng.

**Sửa:** khoá `(i + j, -i)`.

### Lỗi 4 — BFS đẩy con dưới từ **mọi** ô (trùng ô, thiếu `visited`)

In [26]:
def wrong_bfs_dup(nums, cap=50):
    R = len(nums)
    res = []
    q = deque([(0, 0)])
    while q and len(res) < cap:                    # cap để không chạy quá lâu
        i, j = q.popleft()
        res.append(nums[i][j])
        if i + 1 < R and j < len(nums[i + 1]):     # SAI: đẩy con dưới từ mọi ô
            q.append((i + 1, j))
        if j + 1 < len(nums[i]):
            q.append((i, j + 1))
    return res

print("sai :", wrong_bfs_dup(ex1), "-> độ dài", len(wrong_bfs_dup(ex1)))
print("đúng:", Solution().findDiagonalOrder(ex1))

sai : [1, 4, 2, 7, 5, 5, 3, 8, 8, 6, 8, 6, 6, 9, 9, 9, 9, 9, 9] -> độ dài 19
đúng: [1, 4, 2, 7, 5, 3, 8, 6, 9]


**Vì sao sai:** ô `(1, 1)` vừa là con dưới của `(0, 1)` vừa là con phải của `(1, 0)` ⇒ bị đẩy hai lần; càng xuống sâu số bản sao càng tăng (theo hệ số tổ hợp). Kết quả dài 19 thay vì 9, chứa nhiều phần tử lặp.

**Sửa:** chỉ ô ở **cột 0** mới sinh con dưới, và đẩy con dưới **trước** con phải. Khi đó đồ thị là cây, không cần `visited`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đường chéo **phụ** (↗/↙) ⇔ `i + j` hằng số; đường chéo **chính** (↘) ⇔ `i − j` hằng số. Thấy chữ "diagonal" ⇒ nghĩ ngay đến khoá này.
- Mảng răng cưa / thưa ⇒ **chỉ duyệt ô thật**, gom theo khoá.
- Thứ tự trong nhóm: chọn **chiều duyệt** phù hợp để khỏi phải sort.
- Sort theo nhiều tiêu chí, có tiêu chí giảm ⇒ khoá tuple với **dấu trừ**.
- BFS trên lưới đường chéo: cần cấu trúc **cây** (mỗi ô một cha) để khỏi trùng.
- Test nhanh: một cột `[[1],[2],[3]]`, một hàng, hàng giữa ngắn.

## 15. Biến thể của bài

1. **LeetCode 498 Diagonal Traverse**: ma trận chữ nhật, đổi chiều **zigzag** xen kẽ mỗi đường chéo.
2. **LeetCode 1329 Sort the Matrix Diagonally**: gom theo `i − j`, sort từng đường chéo chính.
3. **LeetCode 766 Toeplitz Matrix**: mọi đường chéo chính có cùng giá trị ⇒ so `a[i][j] == a[i−1][j−1]`.
4. **Đường chéo trên-xuống** (trong mỗi đường chéo đi từ hàng nhỏ đến lớn) ⇒ khoá `(i + j, i)`.
5. **N-Queens** (51): kiểm tra hai đường chéo bằng tập `i + j` và `i − j`.

Code biến thể 1 (zigzag trên ma trận chữ nhật, tận dụng gom nhóm):

In [27]:
def diagonal_zigzag(mat):
    groups = defaultdict(list)
    for i in range(len(mat) - 1, -1, -1):
        for j, x in enumerate(mat[i]):
            groups[i + j].append(x)                # trong nhóm: dưới -> trên
    res = []
    for d in range(len(groups)):
        res.extend(groups[d] if d % 2 == 0 else groups[d][::-1])
    return res

print(diagonal_zigzag([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), "(LeetCode 498 ví dụ: [1,2,4,7,5,3,6,8,9])")

[1, 2, 4, 7, 5, 3, 6, 8, 9] (LeetCode 498 ví dụ: [1,2,4,7,5,3,6,8,9])


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét ma trận ảo | Mọi `d`, mọi `i`, kiểm tra `j` hợp lệ | `O(R·D)` | `O(N)` | Ma trận gần chữ nhật, kiểm chứng |
| 2. Gom nhóm `i+j` | Duyệt hàng dưới lên, `groups[i+j].append` | `O(N)` | `O(N)` | Mặc định, nhanh, gọn |
| 3. Sort / heap `(i+j, −i)` | Khoá tuple xác định thứ tự | `O(N log N)` | `O(N)` | Dễ nghĩ nhất, ít bẫy |
| 4. BFS từ `(0,0)` | Cây: dưới (chỉ cột 0) rồi phải | `O(N)` | `O(N)` | Khi quen BFS theo tầng |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().findDiagonalOrder([[1, 2], [3], [4, 5, 6]])` trả về gì?

### Bài 2 (tính tay)
Với ví dụ 2, liệt kê các nhóm `groups[d]` (sau khi duyệt dưới lên) cho `d = 0..8`.

### Bài 3 (điền TODO)
Hoàn thiện `diagonals(nums)` trả về **list các đường chéo** (mỗi đường chéo là một list theo thứ tự dưới-lên).

### Bài 4 (tự viết lại)
Viết `diag_top_down(nums)`: giống đề nhưng trong mỗi đường chéo đi **từ trên xuống** (hàng nhỏ trước). Dùng BFS hoặc gom nhóm.

### Bài 5 (mini challenge)
Viết `anti_main_diagonal_sums(nums)`: trả về dict `{d: tổng các phần tử trên đường chéo chính d = j − i}` cho mảng răng cưa, và `max_main_diag(nums)` trả về `d` có tổng lớn nhất (hoà thì `d` nhỏ nhất).

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def diagonals(nums):
    groups = defaultdict(list)
    # TODO: duyệt hàng từ dưới lên, gom theo i + j
    # TODO: trả về [groups[d] for d in range(len(groups))]
    pass

# Bài 4 — TODO
def diag_top_down(nums):
    # TODO: duyệt hàng từ trên xuống và gom theo i + j (thứ tự tự nhiên là trên -> dưới)
    pass

# Bài 5 — TODO
def anti_main_diagonal_sums(nums):
    # TODO: sums[j - i] += x
    pass

def max_main_diag(nums):
    # TODO: dùng anti_main_diagonal_sums, chọn d có tổng lớn nhất, hoà lấy d nhỏ
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ghi `i + j`: `1(0) 2(1) / 3(1) / 4(2) 5(3) 6(4)`. (2) Nhóm: `d0=[1]`, `d1=[3,2]`, `d2=[4]`, `d3=[5]`, `d4=[6]`. (3) `[1, 3, 2, 4, 5, 6]`.

**Bài 2.** (1) Hàng 4 cho `d4..d8 = 12..16`; hàng 3 cho `d3..d5 = 9..11`. (2) Hàng 2: `d2 += 8`; hàng 1: `d1 += 6`, `d2 += 7`; hàng 0: `d0..d4 += 1..5`. (3) `d0=[1]`, `d1=[6,2]`, `d2=[8,7,3]`, `d3=[9,4]`, `d4=[12,10,5]`, `d5=[13,11]`, `d6=[14]`, `d7=[15]`, `d8=[16]`.

**Bài 3.** (1) Như `Solution`, nhưng không nối. (2) `range(len(groups))`. (3) So `sum(diagonals(...), [])` với `Solution`.

**Bài 4.** (1) Duyệt `i` tăng ⇒ nhóm theo `i` tăng. (2) Hoặc sort khoá `(i + j, i)`. (3) Kiểm tra: kết quả bằng đảo từng nhóm của bài 3.

**Bài 5.** (1) `sums = defaultdict(int)`, `sums[j - i] += x`. (2) `d` có thể âm. (3) `max(sorted(sums), key=lambda d: sums[d])` — hoặc `min(sums, key=lambda d: (-sums[d], d))`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().findDiagonalOrder([[1, 2], [3], [4, 5, 6]]) == [1, 3, 2, 4, 5, 6]

# Bài 3 (dùng luôn cho bài 2)
def diagonals_sol(nums):
    groups = defaultdict(list)
    for i in range(len(nums) - 1, -1, -1):
        for j, x in enumerate(nums[i]):
            groups[i + j].append(x)
    return [groups[d] for d in range(len(groups))]

# Bài 2
assert diagonals_sol(ex2) == [[1], [6, 2], [8, 7, 3], [9, 4], [12, 10, 5], [13, 11], [14], [15], [16]]

# Bài 4
def diag_top_down_sol(nums):
    groups = defaultdict(list)
    for i, row in enumerate(nums):
        for j, x in enumerate(row):
            groups[i + j].append(x)
    return [x for d in range(len(groups)) for x in groups[d]]

# Bài 5
def anti_main_diagonal_sums_sol(nums):
    sums = defaultdict(int)
    for i, row in enumerate(nums):
        for j, x in enumerate(row):
            sums[j - i] += x
    return dict(sums)

def max_main_diag_sol(nums):
    sums = anti_main_diagonal_sums_sol(nums)
    return min(sums, key=lambda d: (-sums[d], d))

assert anti_main_diagonal_sums_sol(ex1) == {0: 15, 1: 8, 2: 3, -1: 12, -2: 7}
assert max_main_diag_sol(ex1) == 0

rng = random.Random(3)
for _ in range(300):
    nums = rand_jagged(rng, rng.randint(1, 30), 8)
    assert sum(diagonals_sol(nums), []) == Solution().findDiagonalOrder(nums)
    assert diag_top_down_sol(nums) == [x for g in diagonals_sol(nums) for x in reversed(g)]
    cells = sorted((i + j, i, x) for i, row in enumerate(nums) for j, x in enumerate(row))
    assert diag_top_down_sol(nums) == [x for _, _, x in cells]
    s = anti_main_diagonal_sums_sol(nums)
    assert sum(s.values()) == sum(map(sum, nums))
    best = max_main_diag_sol(nums)
    assert all(s[best] > s[d] or (s[best] == s[d] and best <= d) for d in s)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template gom theo đường chéo:**

```python
from collections import defaultdict
groups = defaultdict(list)
for i in range(len(a) - 1, -1, -1):      # dưới lên nếu cần "dưới -> trên" trong nhóm
    for j, x in enumerate(a[i]):         # chỉ ô thật (mảng răng cưa)
        groups[i + j].append(x)          # i - j cho đường chéo chính
ans = [x for d in range(len(groups)) for x in groups[d]]
```

**Template sort khoá tuple:** `sorted((i + j, -i, x) ...)`.

**Template BFS cây đường chéo:** từ `(i, j)` đẩy `(i+1, 0)` nếu `j == 0`, rồi `(i, j+1)`.

**Checklist:**
- [ ] Không dùng `len(nums[0])` làm số cột.
- [ ] Trong mỗi đường chéo: **dưới lên** (hàng lớn trước).
- [ ] Khoá sort là `(i + j, -i)`, không phải theo giá trị.
- [ ] BFS không đẩy trùng (con dưới chỉ từ cột 0).
- [ ] Độ phức tạp theo `N` (tổng phần tử), không theo `R × max_len`.
- [ ] Đã thử một cột, một hàng, hàng giữa ngắn.